# Phase 9A — full-range MPRA activity dataset and frozen splits

**Run in Kaggle with Internet ON, CPU, and at least ~5 GB free disk.** Attach these three saved outputs as Inputs (ZIP files or Kaggle-unpacked datasets both work):

1. `[1] enhancer_audit_v1_outputs.zip` — Stage 1 manifest with all 3,268 legacy sequences.
2. `4_enhancer_phase4_external_dataset.zip` — the Phase 4 dataset and audit.
3. `7_enhancer_phase7_chromosome_results.zip` — its protocol and frozen chromosome assignment are verified even if the provenance JSON has a different byte checksum.

The notebook downloads the pinned processed ENCODE BED files and UCSC `hg38.2bit` if they are absent. For offline reruns, additionally attach exact files `ENCFF774DYO.bed.gz`, `ENCFF631LHA.bed.gz`, and `hg38.2bit`; their MD5 hashes are checked. No Phase 5/6 checkpoints or GPU are needed.

This reconstructs the *full continuous activity range* before Phase 4's extreme-quintile selection. It uses the same design families and legacy sequence screen. Two prespecified protocols are saved: grouped locus holdout (9A) and the Phase 7 chromosome partition (9B). Exact/RC sequence duplicates crossing 9B splits are quarantined from the lower-priority side (test > validation > train). It does not fit a model, examine test outcomes to alter splits, or claim these ENCODE candidate elements are a general enhancer gold standard.

**Download the final `enhancer_phase9a_full_activity_dataset.zip` from Kaggle Output.** The next GPU notebook will train regression baselines using these frozen splits.


In [3]:
# 0 — Locked sources, protocol and checksum helpers.
from pathlib import Path
from collections import Counter
import csv, gzip, hashlib, io, json, math, os, re, subprocess, sys, urllib.request, zipfile
import numpy as np
import pandas as pd

WORK=Path('/kaggle/working/enhancer_phase9a'); WORK.mkdir(parents=True,exist_ok=True)
INPUT=Path('/kaggle/input')
SOURCE_SPECS={
 'hg38.2bit': {'md5':'dcc3ea27079aa6dc3f9deccd7275e0f8', 'urls':[
   'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.2bit',
   'https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.2bit']},
 'ENCFF774DYO.bed.gz': {'md5':'87b3fb363f64aeb2b5f893ff98b47db3', 'urls':[
   'https://www.encodeproject.org/files/ENCFF774DYO/@@download/ENCFF774DYO.bed.gz']},
 'ENCFF631LHA.bed.gz': {'md5':'847512c673c75874e84200f84899f880', 'urls':[
   'https://www.encodeproject.org/files/ENCFF631LHA/@@download/ENCFF631LHA.bed.gz']}}
ASSAYS={
 'HepG2':{'file':'ENCFF774DYO.bed.gz','prefix':'DNasePeakNoPromoter','accession':'ENCSR022GQD'},
 'K562':{'file':'ENCFF631LHA.bed.gz','prefix':'peak','accession':'ENCSR382BVV'}}
EXPECTED={
 'stage1_candidate_manifest.csv':'e295155300fafc34528bc1868c6fad6ec78f3912c5b07e034710e81c3b4ac4b3',
 'phase4_audit_and_provenance.json':None,
 'phase4_external_relative_activity_all_cells.csv':'b0bf4a433f1e0af74d95b8747b9312538734f70c425c8379628cafef0c947d2a',
 'phase7_provenance.json':'d40065eeefebd21da693afdcf63c536f9d0d69840a0159f192b1b05f010e2347'}
EXPECTED_PHASE4_PROTOCOL='e49fe4d57dda904bffe9fa5a47d8c8e397668cbee71f24e3b0152fd1781b8705'
EXPECTED_PHASE7_PROTOCOL='b92b384f8728b946f2167fb74cf4c8a7611ea658a5da758ca966d6913d564955'
CONFIG={'name':'phase9a_full_range_mpra_v1','reference':'UCSC hg38','length':200,
 'source_md5':{k:v['md5'] for k,v in SOURCE_SPECS.items()},'assays':ASSAYS,
 'legacy_similarity_k':15,'locus_split_seed':20261009,
 'locus_test_fraction_groups':0.20,'locus_validation_fraction_remaining_groups':0.20,
 'chromosome_collision_priority':['test','validation','train'],
 'phase4_protocol':EXPECTED_PHASE4_PROTOCOL,'phase7_protocol':EXPECTED_PHASE7_PROTOCOL,
 'stage1_manifest_sha256':EXPECTED['stage1_candidate_manifest.csv']}
def sha(raw):return hashlib.sha256(raw).hexdigest()
def filehash(path,algorithm='sha256'):
 h=hashlib.new(algorithm)
 with open(path,'rb') as f:
  for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
 return h.hexdigest()
PROTOCOL_HASH=sha(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode())
print('Phase 9A protocol:',PROTOCOL_HASH)


Phase 9A protocol: 5e438773a3f9c74cf006d183bcfc30daabd3d85bdbfefd47acfa48dc023e450a


In [4]:
# 1 — Resolve saved Kaggle Inputs; Phase 7 has a pinned, auditable fallback.
def read_locked_member(name, verify_sha=True):
    expected = EXPECTED[name] if verify_sha else None
    found = []
    if INPUT.exists():
        for p in INPUT.rglob(name):
            if p.is_file():
                found.append((p.read_bytes(), str(p)))
        for p in INPUT.rglob('*.zip'):
            try:
                with zipfile.ZipFile(p) as z:
                    members = [m for m in z.namelist()
                               if m == name or m.endswith('/' + name)]
                    if len(members) == 1:
                        found.append((z.read(members[0]), str(p) + '!' + members[0]))
            except (zipfile.BadZipFile, OSError):
                pass

    if not found:
        raise FileNotFoundError(
            f'Missing {name}. Attach the saved Phase 1, Phase 4, and Phase 7 outputs.'
        )
    matching = [(raw, source) for raw, source in found
                if expected is None or sha(raw) == expected]
    if not matching:
        raise ValueError(
            f'Found {name}, but its SHA256 differs from the pinned version: '
            + str([(source, sha(raw)) for raw, source in found])
        )
    if len({sha(raw) for raw, _ in matching}) != 1:
        raise RuntimeError(f'Ambiguous versions of {name}')
    print('Verified input', name, 'from', matching[0][1])
    return matching[0][0]

legacy_raw = read_locked_member('stage1_candidate_manifest.csv')
legacy = pd.read_csv(io.BytesIO(legacy_raw), dtype=str, keep_default_na=False)
assert len(legacy) == 3268 and legacy.record_id.is_unique
assert legacy.sequence_sha256.is_unique
assert legacy.sequence.str.fullmatch('[ACGT]{200}').all()
assert all(sha(s.encode()) == h
           for s, h in zip(legacy.sequence, legacy.sequence_sha256))
assert set(legacy.role) == {
    'train_candidate', 'legacy_test', 'excluded_related_to_legacy_test'
}
legacy_sequences = legacy.sequence.tolist()

audit4 = json.loads(read_locked_member('phase4_audit_and_provenance.json'))
assert audit4['protocol_sha256'] == EXPECTED_PHASE4_PROTOCOL
phase4_raw = read_locked_member('phase4_external_relative_activity_all_cells.csv')
assert audit4['output_sha256']['phase4_external_relative_activity_all_cells.csv'] == sha(phase4_raw)
phase4 = pd.read_csv(io.BytesIO(phase4_raw), dtype={
    'record_id': str, 'sequence': str, 'sequence_sha256': str
})
assert len(phase4) == 30340 and phase4.record_id.is_unique
assert set(phase4.cell_line) == {'HepG2', 'K562'}

# The Phase 7 notebook froze these exact chromosome sets. If its saved output
# is attached, authenticate it and require the same assignment. Otherwise use
# the embedded assignment; no Phase 7 result file is needed to build Phase 9A.
FROZEN_CHROMOSOMES = {
    'train': ['chr11', 'chr13', 'chr14', 'chr16', 'chr17', 'chr18',
              'chr19', 'chr2', 'chr21', 'chr22', 'chr3', 'chr6',
              'chr8', 'chrX', 'chrY'],
    'validation': ['chr10', 'chr15', 'chr20', 'chr5'],
    'test': ['chr1', 'chr12', 'chr4', 'chr7', 'chr9']
}
phase7_raw = read_locked_member('phase7_provenance.json', verify_sha=False)
phase7_observed_sha = sha(phase7_raw)
if phase7_observed_sha != EXPECTED['phase7_provenance.json']:
    print('Phase 7 JSON has a different byte SHA256 from the archived copy.')
    print('Observed SHA256:', phase7_observed_sha)
    print('Checking its frozen protocol and chromosome assignment instead.')
provenance7 = json.loads(phase7_raw)
assert provenance7['protocol_sha256'] == EXPECTED_PHASE7_PROTOCOL, (
    'Phase 7 protocol mismatch: ' + str(provenance7.get('protocol_sha256'))
)
assert provenance7['config']['phase4_protocol'] == EXPECTED_PHASE4_PROTOCOL
assert {k: sorted(v) for k, v in provenance7['split_chromosomes'].items()} == {
    k: sorted(v) for k, v in FROZEN_CHROMOSOMES.items()
}, 'Phase 7 chromosome assignment differs from the frozen split.'
CHROMOSOMES = provenance7['split_chromosomes']
assert set(CHROMOSOMES) == {'train', 'validation', 'test'}
assert len(set().union(*(set(v) for v in CHROMOSOMES.values()))) == 24
assert sum(len(v) for v in CHROMOSOMES.values()) == 24
print('Legacy sequences:', len(legacy_sequences),
      'Phase 4 selected:', len(phase4),
      'Phase 7 chromosomes:', CHROMOSOMES)


Verified input stage1_candidate_manifest.csv from /kaggle/input/datasets/aryanaditta/enhancer-audit-v1-outputs-zip/stage1_candidate_manifest.csv
Verified input phase4_audit_and_provenance.json from /kaggle/input/datasets/aryanaditta/4-enhancer-phase4-external-dataset-zip/phase4_audit_and_provenance.json
Verified input phase4_external_relative_activity_all_cells.csv from /kaggle/input/datasets/aryanaditta/4-enhancer-phase4-external-dataset-zip/phase4_external_relative_activity_all_cells.csv
Verified input phase7_provenance.json from /kaggle/input/datasets/aryanaditta/7-enhancer-phase7-chromosome-results-zip/phase7_provenance.json
Phase 7 JSON has a different byte SHA256 from the archived copy.
Observed SHA256: 49d955acd40448e4ce415c1ff38df2def7fa80ec82bfb2386c8d9fa92e792968
Checking its frozen protocol and chromosome assignment instead.
Legacy sequences: 3268 Phase 4 selected: 30340 Phase 7 chromosomes: {'test': ['chr1', 'chr12', 'chr4', 'chr7', 'chr9'], 'train': ['chr11', 'chr13', 'chr

In [5]:
# 2 — Retrieve pinned raw inputs and load the hg38 reference.
def get_source(name):
 spec=SOURCE_SPECS[name]; candidates=[]
 local=WORK/name
 if local.exists():candidates.append(local)
 if INPUT.exists():candidates+=sorted(p for p in INPUT.rglob(name) if p.is_file())
 for p in candidates:
  if filehash(p,'md5')==spec['md5']:
   print('Verified raw file:',p);return p
 if candidates:raise RuntimeError(name+' exists but has the wrong MD5; remove it or attach exact pinned bytes')
 errors=[]
 for url in spec['urls']:
  temp=WORK/(name+'.part')
  try:
   req=urllib.request.Request(url,headers={'User-Agent':'enhancer-phase9a/1.0'})
   with urllib.request.urlopen(req,timeout=120) as response,open(temp,'wb') as out:
    for block in iter(lambda:response.read(8*1024*1024),b''):out.write(block)
   if filehash(temp,'md5')!=spec['md5']:raise ValueError('Downloaded MD5 mismatch')
   os.replace(temp,local);print('Downloaded',name,local.stat().st_size,'bytes');return local
  except Exception as e:
   temp.unlink(missing_ok=True);errors.append(str(e)[:180])
 raise RuntimeError('Cannot retrieve '+name+'; attach exact file or enable Internet. '+str(errors))
try:import py2bit
except ImportError:
 subprocess.check_call([sys.executable,'-m','pip','-q','install','py2bit==0.3.3'])
 import py2bit
paths={name:get_source(name) for name in SOURCE_SPECS}
genome=py2bit.open(str(paths['hg38.2bit']))
chrom_sizes=genome.chroms()
assert chrom_sizes['chr1']>240_000_000 and chrom_sizes['chrX']>150_000_000


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.4 MB/s eta 0:00:00
Downloaded hg38.2bit 835393456 bytes
Downloaded ENCFF774DYO.bed.gz 2545246 bytes
Downloaded ENCFF631LHA.bed.gz 3459202 bytes


In [6]:
# 3 — Reconstruct all valid paired-orientation 200 bp assay elements.
COL=['chrom','start','end','name','score','strand','log2fc','input_count','output_count','minus_log10_p','minus_log10_q']
CANONICAL=re.compile(r'chr(?:[1-9]|1[0-9]|2[0-2]|X|Y)')
def parse_assay(cell,spec):
 audit=Counter();rows=[];want=spec['prefix']
 pattern=re.compile(re.escape(want)+r'\d+(?:_Reversed:)?')
 with gzip.open(paths[spec['file']],'rt') as f:
  for line_no,line in enumerate(f,1):
   if not line.strip() or line.startswith(('#','track','browser')):continue
   fields=line.rstrip('\n').split('\t')
   if len(fields)!=11:raise ValueError(f'{cell} line {line_no}: expected BED6+5, got {len(fields)} columns')
   rec=dict(zip(COL,fields));audit['rows_total']+=1
   if not pattern.fullmatch(rec['name']):audit['other_design_name']+=1;continue
   chrom=rec['chrom'];start=int(rec['start']);end=int(rec['end'])
   if not CANONICAL.fullmatch(chrom):audit['noncanonical_chrom']+=1;continue
   if end-start!=200 or start<0 or end>chrom_sizes[chrom]:audit['non_200bp']+=1;continue
   if rec['strand'] not in ('+','-'):audit['unknown_strand']+=1;continue
   if rec['minus_log10_p']!='-1' or rec['minus_log10_q']!='-1':
    raise ValueError('P/Q fields changed from the pinned release; review before proceeding')
   fc=float(rec['log2fc']);ic=float(rec['input_count']);oc=float(rec['output_count'])
   if not all(map(math.isfinite,(fc,ic,oc))) or ic<=0 or oc<=0:
    audit['unreliable_counts']+=1;continue
   rows.append((chrom,start,end,rec['name'],rec['strand'],fc,ic,oc));audit['candidate_rows']+=1
 df=pd.DataFrame(rows,columns=['chrom','start','end','name','strand','log2fc','input_count','output_count'])
 assert len(df)>1000,(cell,'too few candidates')
 df['base_name']=df.name.str.replace(r'_Reversed:$','',regex=True)
 grouped=df.groupby(['chrom','start','end','base_name'],sort=True,observed=True)
 summary=grouped.agg(n=('log2fc','size'),strands=('strand','nunique'),
  mean_log2_activity=('log2fc','mean'),min_input_count=('input_count','min'),
  min_output_count=('output_count','min')).reset_index()
 good=summary[(summary.n==2)&(summary.strands==2)].copy()
 audit['incomplete_or_ambiguous_orientation']=len(summary)-len(good)
 audit['complete_two_orientation_elements']=len(good)
 assert len(good)>1000,(cell,'too few paired elements')
 good['cell_line']=cell
 print(cell,'paired elements:',len(good));return good,dict(audit)
assayed={};audits={}
for cell,spec in ASSAYS.items():assayed[cell],audits[cell]=parse_assay(cell,spec)
assert len(assayed['HepG2'])==audit4['thresholds']['HepG2']['population_size']
assert len(assayed['K562'])==audit4['thresholds']['K562']['population_size']
for cell,df in assayed.items():
 lo,hi=df.mean_log2_activity.quantile([.2,.8])
 assert np.isclose(lo,audit4['thresholds'][cell]['low_max'],atol=1e-10)
 assert np.isclose(hi,audit4['thresholds'][cell]['high_min'],atol=1e-10)
 print(cell,'Phase 4 quintiles reproduced; all middle activity rows retained')


HepG2 paired elements: 64190
K562 paired elements: 85442
HepG2 Phase 4 quintiles reproduced; all middle activity rows retained
K562 Phase 4 quintiles reproduced; all middle activity rows retained


In [7]:
# 4 — Extract the reference sequence and apply the Phase 4 legacy similarity screen.
TRANS=str.maketrans('ACGT','TGCA')
def rc(s):return s.translate(TRANS)[::-1]
def words(seq,k):return (seq[i:i+k] for i in range(len(seq)-k+1))
legacy_exact={min(s,rc(s)) for s in legacy_sequences}
legacy_words=set()
for seq in legacy_sequences:
 legacy_words.update(words(seq,15));legacy_words.update(words(rc(seq),15))
print('Legacy distinct 15-mers:',len(legacy_words))
clean=[]
for cell,df in assayed.items():
 counts=Counter();rows=[]
 for row in df.itertuples(index=False):
  seq=genome.sequence(row.chrom,int(row.start),int(row.end)).upper()
  if not re.fullmatch('[ACGT]{200}',seq):counts['non_ACGT_reference']+=1;continue
  if min(seq,rc(seq)) in legacy_exact:counts['exact_or_RC_legacy']+=1;continue
  if any(w in legacy_words for w in words(seq,15)) or any(w in legacy_words for w in words(rc(seq),15)):
   counts['shared_15mer_with_legacy']+=1;continue
  record_id=sha((cell+'|'+row.chrom+'|'+str(row.start)+'|'+str(row.end)+'|'+row.base_name).encode())[:20]
  rows.append({'record_id':record_id,'cell_line':cell,'chrom':row.chrom,'start':int(row.start),
   'end':int(row.end),'design_name':row.base_name,'sequence':seq,'sequence_sha256':sha(seq.encode()),
   'mean_log2_activity':float(row.mean_log2_activity),'min_input_count':float(row.min_input_count),
   'min_output_count':float(row.min_output_count)})
 x=pd.DataFrame(rows)
 assert len(x)>1000 and x.record_id.is_unique
 # Check the earlier Phase 4 rows before new full-range deduplication.
 previous=phase4.loc[phase4.cell_line.eq(cell),['record_id','sequence_sha256','mean_log2_activity']]
 matched=previous.merge(x,on='record_id',how='left',validate='one_to_one',suffixes=('_phase4','_new'),indicator=True)
 assert matched._merge.eq('both').all(),(cell,'Phase 4 input rows missing before deduplication')
 assert matched.sequence_sha256_phase4.eq(matched.sequence_sha256_new).all()
 assert np.allclose(matched.mean_log2_activity_phase4,matched.mean_log2_activity_new,rtol=0,atol=1e-12)
 # Resolve repeated sequences by genomic order alone, without selecting by target.
 x=x.sort_values(['sequence_sha256','chrom','start','record_id'])
 before=len(x);x=x.drop_duplicates('sequence_sha256',keep='first')
 counts['within_cell_duplicate_sequences']=before-len(x)
 counts['phase4_rows_preserved_after_full_dedup']=int(x.record_id.isin(previous.record_id).sum())
 x=x.sort_values(['chrom','start','record_id']).reset_index(drop=True)
 assert x.record_id.is_unique and x.sequence_sha256.is_unique
 audits[cell].update(counts);audits[cell]['full_range_rows']=len(x)
 clean.append(x)
 print(cell,'full-range rows:',len(x),'screen audit:',dict(counts))
full=pd.concat(clean,ignore_index=True).sort_values(['cell_line','chrom','start','record_id']).reset_index(drop=True)
full['gc_count']=full.sequence.str.count('G')+full.sequence.str.count('C')
full['cpg_count']=full.sequence.str.count('CG')
full['gc_fraction']=full.gc_count/200.
assert not set(min(s,rc(s)) for s in full.sequence)&legacy_exact
assert len(full)>len(phase4) and set(full.cell_line)=={'HepG2','K562'}
# A full-range duplicate can resolve to another assayed locus than the Phase 4 subset.
# This count documents the difference; verification of all old rows occurred before deduplication.
phase4_final_overlap=int(full.record_id.isin(phase4.record_id).sum())
print('Phase 4 retained rows also present after full-range dedup:',phase4_final_overlap,'/',len(phase4))


Legacy distinct 15-mers: 1177946
HepG2 full-range rows: 33324 screen audit: {'shared_15mer_with_legacy': 30863, 'exact_or_RC_legacy': 3, 'within_cell_duplicate_sequences': 0, 'phase4_rows_preserved_after_full_dedup': 13142}
K562 full-range rows: 43509 screen audit: {'shared_15mer_with_legacy': 41932, 'exact_or_RC_legacy': 1, 'within_cell_duplicate_sequences': 0, 'phase4_rows_preserved_after_full_dedup': 17198}
Phase 4 retained rows also present after full-range dedup: 30340 / 30340


In [8]:
# 5 — Locus groups across both cells; split exact/RC components together for protocol 9A.
loci=full[['chrom','start','end']].drop_duplicates().sort_values(['chrom','start','end'])
lookup={};counter=0
for chrom,grp in loci.groupby('chrom',sort=True):
 current_end=-1;group_id=None
 for row in grp.itertuples(index=False):
  if int(row.start)>=current_end:
   counter+=1;group_id=f'{chrom}_locus_{counter:07d}';current_end=int(row.end)
  else:current_end=max(current_end,int(row.end))
  lookup[(row.chrom,int(row.start),int(row.end))]=group_id
full['locus_group_id']=[lookup[(r.chrom,int(r.start),int(r.end))] for r in full.itertuples(index=False)]
assert full.groupby(['chrom','start','end']).locus_group_id.nunique().max()==1
parent={g:g for g in full.locus_group_id.unique()}
def find(g):
 while parent[g]!=g:
  parent[g]=parent[parent[g]];g=parent[g]
 return g
def union(a,b):
 a,b=find(a),find(b)
 if a!=b:parent[max(a,b)]=min(a,b)
seen={}
for seq,g in full[['sequence','locus_group_id']].itertuples(index=False):
 key=min(seq,rc(seq))
 if key in seen:union(g,seen[key])
 else:seen[key]=g
full['split_group_id']=full.locus_group_id.map(find)
print('Locus groups:',full.locus_group_id.nunique(),'exact/RC linked components:',full.split_group_id.nunique())


Locus groups: 64709 exact/RC linked components: 64709


In [9]:
# 6 — Freeze 9A group holdout and 9B chromosome holdout before model training.
from sklearn.model_selection import GroupShuffleSplit
groups=np.array(sorted(full.split_group_id.unique()))
gss=GroupShuffleSplit(n_splits=1,test_size=CONFIG['locus_test_fraction_groups'],random_state=CONFIG['locus_split_seed'])
remaining,test_ix=next(gss.split(groups,groups=groups))
remain=groups[remaining];test_groups=set(groups[test_ix])
gss2=GroupShuffleSplit(n_splits=1,test_size=CONFIG['locus_validation_fraction_remaining_groups'],random_state=CONFIG['locus_split_seed']+1)
train_ix,val_ix=next(gss2.split(remain,groups=remain))
val_groups=set(remain[val_ix]);train_groups=set(remain[train_ix])
assert not ((test_groups&val_groups)|(test_groups&train_groups)|(val_groups&train_groups))
full['split_9a']=np.where(full.split_group_id.isin(test_groups),'test',np.where(full.split_group_id.isin(val_groups),'validation','train'))
chrom_map={chrom:split for split,chroms in CHROMOSOMES.items() for chrom in chroms}
full['split_9b']=full.chrom.map(chrom_map)
assert full.split_9b.notna().all()
# A sequence occurring on multiple chromosomes can cross the frozen partition.
# Preserve Phase 7 chromosomes and quarantine the lower-priority copies from 9B.
priority={'test':0,'validation':1,'train':2}
full['sequence_canonical_sha256']=[sha(min(s,rc(s)).encode()) for s in full.sequence]
min_priority=full.groupby('sequence_canonical_sha256').split_9b.transform(lambda x:min(priority[v] for v in x))
full['quarantined_9b_exact_rc']=full.split_9b.map(priority).gt(min_priority)
active=full[~full.quarantined_9b_exact_rc]
assert full.groupby('split_group_id').split_9a.nunique().max()==1
assert full.groupby('locus_group_id').split_9a.nunique().max()==1
assert active.groupby('sequence_canonical_sha256').split_9b.nunique().max()==1
assert active.groupby('locus_group_id').split_9b.nunique().max()==1
assert all(active.groupby(['cell_line','split_9b']).size()>100)
assert all(full.groupby(['cell_line','split_9a']).size()>100)
# No overlapping genomic intervals cross a split within a chromosome.
assert full.groupby('locus_group_id').split_9b.nunique().max()==1
for col,frame in [('split_9a',full),('split_9b',active)]:
 print(col,frame.groupby(['cell_line',col]).mean_log2_activity.agg(['size','mean','std']).to_string())
print('Quarantined exact/RC 9B rows:',int(full.quarantined_9b_exact_rc.sum()))


split_9a                        size      mean       std
cell_line split_9a                             
HepG2     test         6725 -0.254196  0.705243
          train       21307 -0.249930  0.706683
          validation   5292 -0.262003  0.692648
K562      test         8661 -0.150064  0.431297
          train       27931 -0.139862  0.436805
          validation   6917 -0.131186  0.434130
split_9b                        size      mean       std
cell_line split_9b                             
HepG2     test         9048 -0.255603  0.710002
          train       18665 -0.247964  0.701671
          validation   5611 -0.263822  0.703026
K562      test        13145 -0.156646  0.429617
          train       23560 -0.133795  0.438825
          validation   6804 -0.132612  0.433283
Quarantined exact/RC 9B rows: 0


In [10]:
# 7 — Save single-cell data, split manifest, protocol/audit and verified ZIP.
DATA_COLS=['record_id','cell_line','chrom','start','end','design_name','sequence','sequence_sha256',
 'mean_log2_activity','min_input_count','min_output_count','gc_count','cpg_count','gc_fraction',
 'locus_group_id','split_group_id','split_9a','split_9b','quarantined_9b_exact_rc']
files=[]
for cell,part in full.groupby('cell_line',sort=True):
 p=WORK/f'phase9_full_activity_{cell.lower()}.csv';part[DATA_COLS].to_csv(p,index=False);files.append(p)
all_file=WORK/'phase9_full_activity_all_cells.csv';full[DATA_COLS].to_csv(all_file,index=False);files.append(all_file)
split_cols=['record_id','cell_line','locus_group_id','split_group_id','sequence_canonical_sha256','split_9a','split_9b','quarantined_9b_exact_rc']
split_file=WORK/'phase9_frozen_splits.csv';full[split_cols].to_csv(split_file,index=False);files.append(split_file)
report={'protocol_hash':PROTOCOL_HASH,'config':CONFIG,'sources':{name:{'md5':filehash(p,'md5'),'sha256':filehash(p)} for name,p in paths.items()},
 'input_sha256':{name:sha(raw) for name,raw in [('stage1_candidate_manifest.csv',legacy_raw),
 ('phase4_external_relative_activity_all_cells.csv',phase4_raw)]},
 'phase4_selected_rows_reproduced_before_full_dedup':len(phase4),'phase4_rows_retained_after_full_dedup':phase4_final_overlap,'phase4_protocol':audit4['protocol_sha256'],
 'phase7_provenance_sha256':sha(phase7_raw) if phase7_raw is not None else None,
 'phase7_assignment_source':'attached provenance verified by protocol and chromosome assignment',
 'phase7_chromosome_assignment':CHROMOSOMES,'assay_audit':audits,
 'full_range_count_by_cell':{k:int(v) for k,v in full.cell_line.value_counts().items()},
 '9b_exact_rc_quarantine_count':int(full.quarantined_9b_exact_rc.sum()),
 'outcome_scope':'Within-assay log2 output/input activity averaged over two orientations for 200 bp tested genomic candidate sequences.',
 'caveats':['HepG2 DNasePeakNoPromoter and K562 peak are different design families.',
  'No measured replicate-level variance or credible interval is inferred from the BED P/Q fields (-1).',
  'Legacy shared-15-mer screen is local and cannot prove absence of all homology.',
  'A 9B quarantined row remains in the dataset but must be excluded from every 9B fit and evaluation.',
  'Exact/RC linking does not guarantee removal of all near-duplicate sequences across splits.']}
report['output_sha256']={p.name:filehash(p) for p in files}
report_file=WORK/'phase9a_audit_and_provenance.json';report_file.write_text(json.dumps(report,sort_keys=True,indent=2)+'\n');files.append(report_file)
zip_path=Path('/kaggle/working/enhancer_phase9a_full_activity_dataset.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in files:z.write(p,p.name)
with zipfile.ZipFile(zip_path) as z:
 assert z.testzip() is None
 for name,digest in report['output_sha256'].items():assert sha(z.read(name))==digest,name
print('READY:',zip_path,'bytes:',zip_path.stat().st_size,'protocol:',PROTOCOL_HASH)
print('Download this ZIP and save it as a Kaggle Dataset for the Phase 9 GPU regression notebook.')


READY: /kaggle/working/enhancer_phase9a_full_activity_dataset.zip bytes: 28990918 protocol: 5e438773a3f9c74cf006d183bcfc30daabd3d85bdbfefd47acfa48dc023e450a
Download this ZIP and save it as a Kaggle Dataset for the Phase 9 GPU regression notebook.
